In [1]:
from func import *

In [2]:
train_df = load_csvs_from_ftp_to_df(provided_local_dir="/")

In [3]:
# Strictly positive features
positive_col_list = ["BedroomsTotal",
                     "BathroomsTotalInteger",
                     "LotSizeAcres",
                     "LotSizeArea",
                     "LotSizeSquareFeet",
                     "LivingArea"]

# Non-negative features
non_negative_col_list = ["ParkingTotal"]

# Drop ID-like features
default_drop_col = ["ListingId",
                    "ListingKey",
                    "ListingKeyNumeric",
                    "ListPrice",
                    "OriginalListPrice"]

# Remove columns with over 30% missing
missing_threshold = 0.8

In [4]:
(train_df_clean,
 knn_model,
 train_df_ref,
 reference_col_list,
 k_means_model,
 col_drop_list,
 scaler,
 cols_with_na,
 train_df_cluster_ref,
     save_name) = pre_process(train_df,
                              price_col="ClosePrice",
                              default_threshold=missing_threshold,
                              col_drop=default_drop_col,
                              positive_col_list=positive_col_list,
                              non_negative_col_list=non_negative_col_list,
                              flag_col_list=["Flooring"],
                              yn_col_list=["AttachedGarageYN",
                                           "ViewYN",
                                           "NewConstructionYN",
                                           "PoolPrivateYN",
                                           "FireplaceYN"],
                              knn_k=3,
                              knn_model=None,
                              train_df_ref=None,
                              reference_col_list=None,
                              num_clusters=10,
                              clustering_method="k-means",
                              clustering_model=None,
                              train_df_cluster_ref=None,
                              scaler_method="robust",
                              scaler=None,
                              save_name="processed",
                              train_data=True,
                              save=True,
                              cols_with_na=None
                              )

C:\My_Programs\Python_Project\IDX_Intern_self\func.py:264: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[val + "YN"] = df[val + "YN"].fillna(False)
C:\My_Programs\Python_Project\IDX_Intern_self\func.py:264: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[val + "YN"] = df[val + "YN"].fillna(False)
C:\My_Programs\Python_Project\IDX_Intern_self\func.py:264: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future beh

In [5]:
test_df = load_csvs_from_ftp_to_df(provided_local_dir="/",
                                   date_range=range(12, 13))
col_drop_list.extend(default_drop_col)

In [6]:
test_df_clean, _, _, _, _, _, _, _, _, _ = pre_process(test_df,
                                                 price_col="ClosePrice",
                                                 default_threshold=missing_threshold,
                                                 col_drop=col_drop_list,
                                                 positive_col_list=positive_col_list,
                                                 non_negative_col_list=non_negative_col_list,
                                                 flag_col_list=["Flooring"],
                                                 yn_col_list=["AttachedGarageYN",
                                                              "ViewYN",
                                                              "NewConstructionYN",
                                                               "PoolPrivateYN",
                                                               "FireplaceYN"],
                                                 knn_k=3,
                                                 knn_model=knn_model,
                                                 train_df_ref=train_df_ref,
                                                 reference_col_list=reference_col_list,
                                                 num_clusters=10,
                                                 clustering_method="k-means",
                                                 clustering_model=k_means_model,
                                                train_df_cluster_ref=train_df_cluster_ref,
                                                 scaler_method="robust",
                                                 scaler=scaler,
                                                 save_name=save_name,
                                                 train_data=False,
                                                 save=True,
                                                 cols_with_na=cols_with_na
                                                 )

C:\My_Programs\Python_Project\IDX_Intern_self\func.py:264: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[val + "YN"] = df[val + "YN"].fillna(False)
C:\My_Programs\Python_Project\IDX_Intern_self\func.py:264: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[val + "YN"] = df[val + "YN"].fillna(False)
C:\My_Programs\Python_Project\IDX_Intern_self\func.py:264: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future beh

In [16]:
train_df = pd.read_csv("processed7/train_data.csv")
test_df = pd.read_csv("processed7/test_data.csv")

train_df["logClosePrice"] = np.log1p(train_df["ClosePrice"])
train_df.drop(columns=["ClosePrice"], inplace=True)
test_df["logClosePrice"] = np.log1p(test_df["ClosePrice"])
test_df.drop(columns=["ClosePrice"], inplace=True)

In [17]:
X_train = train_df.drop(columns=["logClosePrice"], axis=1)

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

nunique = X_train[cat_cols].nunique(dropna=False)

low_card_cols = nunique[(nunique <= 20)].index.tolist()
high_card_cols = nunique[(nunique >= 20)].index.tolist()


In [9]:
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor
from scipy.stats import randint, uniform, loguniform
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

In [10]:

# Tuning
en_param_grid = param_grid = {
    "model__alpha": np.logspace(-4, 2, 25),
    "model__l1_ratio": np.linspace(0.05, 0.95, 19),
}

res = grid_tune_with_make_model_pipeline(
    train_df=train_df,
    target_col="logClosePrice",
    model=ElasticNet(max_iter=20000),
    param_grid=param_grid,
    col_drop_list=["ClosePrice"],
    card_threshold=20,
    scoring="r2",
    cv=5
)

print(res["best_score"], res["best_params"])

Fitting 5 folds for each of 475 candidates, totalling 2375 fits
0.8539774944791765 {'model__alpha': 0.01778279410038923, 'model__l1_ratio': 0.05}


In [11]:
alpha = res["best_params"]["model__alpha"]
l_1_ratio = res["best_params"]["model__l1_ratio"]
model = ElasticNet(alpha=alpha,
                   l1_ratio=l_1_ratio)

elastic_net_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20
)
el_metrics = elastic_net_result["Metrics"]
print(el_metrics)

{'R2(log)': 0.8533663534392709, 'R2': 0.736974364341703, 'MAPE': 17.337753957359983, 'MdAPE': 12.501157370461158, 'RMSE': 450054.2028399734, 'MAE': 208813.36998107337, 'Bias(mean residual)': -52519.69801367883, 'APE_95pct': 49.010736866379496, 'APE_99pct': 83.73112526202088, 'APE_max': 243.99478422638836, 'Train_R2(log)': 0.8923797163640554, 'Test_R2(log)': 0.8533663534392709, 'R2_gap': 0.03901336292478452}


In [12]:

xgb_param_dist = {
    "model__max_depth": randint(3, 7),                    # 3–6
    "model__min_child_weight": randint(8, 31),           # 8–30
    "model__learning_rate": loguniform(0.01, 0.08),      # small LR
    "model__n_estimators": randint(400, 1400),
    "model__subsample": uniform(0.6, 0.3),               # 0.6–0.9
    "model__colsample_bytree": uniform(0.5, 0.4),        # 0.5–0.9
    "model__reg_lambda": loguniform(1.0, 50.0),          # strong L2
    "model__reg_alpha": loguniform(1e-4, 2.0),           # mild–moderate L1
}

model = make_model_numeric_only_pipeline(model=XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    n_jobs=1),
 num_cols=num_cols,
num_scaler=None)


xgb_rs = RandomizedSearchCV(
    model,
    param_distributions=xgb_param_dist,
    n_iter=40,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

xgb_rs.fit(train_df.drop("logClosePrice", axis=1), train_df["logClosePrice"])
print(xgb_rs.best_params_)

{'model__colsample_bytree': 0.7844598129752072, 'model__learning_rate': 0.05383345943137039, 'model__max_depth': 6, 'model__min_child_weight': 14, 'model__n_estimators': 1219, 'model__reg_alpha': 1.10973369349242, 'model__reg_lambda': 4.736558858366902, 'model__subsample': 0.755325405158244}


In [13]:
xgb_model = XGBRegressor(n_estimators=xgb_rs.best_params_["model__n_estimators"],
                         learning_rate=xgb_rs.best_params_["model__learning_rate"],
                         max_depth=xgb_rs.best_params_["model__max_depth"],
                         subsample=xgb_rs.best_params_["model__subsample"],
                         colsample_bytree=xgb_rs.best_params_["model__colsample_bytree"],
                         min_child_weight=xgb_rs.best_params_["model__min_child_weight"],
                         reg_lambda=xgb_rs.best_params_["model__reg_lambda"],
                         reg_alpha=xgb_rs.best_params_["model__reg_alpha"])

# xgb_model = XGBRegressor(n_estimators=xgb_rs.best_params_["model__n_estimators"],
#                          learning_rate=xgb_rs.best_params_["model__learning_rate"],
#                          max_depth=xgb_rs.best_params_["model__max_depth"],
#                          subsample=xgb_rs.best_params_["model__subsample"],
#                          colsample_bytree=xgb_rs.best_params_["model__colsample_bytree"],
#                          min_child_weight=xgb_rs.best_params_["model__min_child_weight"],
#                          reg_lambda=xgb_rs.best_params_["model__reg_lambda"],
#                          reg_alpha=xgb_rs.best_params_["model__reg_alpha"]
#                          )

xgb_result = fit_predict(
    train_df=train_df,
    test_df=test_df,
    model=xgb_model,
    col_drop_list=["ClosePrice"],
    target_col="logClosePrice",
    card_threshold=20,
    num_scaler=None,
    num_only=True
)

xgb_metrics = xgb_result["Metrics"]
print(xgb_metrics)

{'R2(log)': 0.923908594513877, 'R2': 0.8792154407761521, 'MAPE': 11.876189831278493, 'MdAPE': 8.171911682004184, 'RMSE': 304980.09242140426, 'MAE': 144033.4989588754, 'Bias(mean residual)': -8259.75706800324, 'APE_95pct': 35.14740246710533, 'APE_99pct': 62.08589875188896, 'APE_max': 211.11192857142882, 'Train_R2(log)': 0.9539733570841914, 'Test_R2(log)': 0.923908594513877, 'R2_gap': 0.03006476257031443}


In [18]:
from sklearn.ensemble import StackingRegressor


en_full = make_model_pipeline(
    model=ElasticNet(alpha=0.01778279410038923,
                     l1_ratio=0.05),
    num_cols=num_cols,
    low_card_cols=low_card_cols,
    high_card_cols=high_card_cols
)

xgb_num = make_model_numeric_only_pipeline(
    model=XGBRegressor(n_estimators=1219,
                       learning_rate=0.05383345943137039,
                       max_depth=6,
                       subsample=0.755325405158244,
                       colsample_bytree=0.7844598129752072,
                       min_child_weight=14,
                       reg_lambda=4.736558858366902,
                       reg_alpha=1.10973369349242),
    num_cols=num_cols,
    num_scaler=None
)

# ---------- Stack them ----------
stack = StackingRegressor(
    estimators=[
        ("en_full", en_full),
        ("xgb_num", xgb_num),
    ],
    final_estimator=XGBRegressor(
        max_depth=2,
        n_estimators=200,
        learning_rate=0.05
    ),
    cv=5,
    n_jobs=-1
)

# Fit on log target
y_train = train_df["logClosePrice"]
stack.fit(X_train, y_train)

# Predict log target
X_test = test_df.drop("logClosePrice", axis=1)
y_test = test_df["logClosePrice"]
y_train_pred = stack.predict(X_train)
y_pred = stack.predict(X_test)

stack_metrics = compute_metrics(y_test, y_pred, y_train, y_train_pred)

In [19]:
stack_metrics

{'R2(log)': 0.9256454618996856,
 'R2': 0.879826371647398,
 'MAPE': 11.72432111154959,
 'MdAPE': 8.031679878048775,
 'RMSE': 304207.8167374893,
 'MAE': 142508.381233087,
 'Bias(mean residual)': -6035.597762281446,
 'APE_95pct': 34.85138057065213,
 'APE_99pct': 60.63978757952128,
 'APE_max': 197.29516071428594,
 'Train_R2(log)': 0.9538091838906688,
 'Test_R2(log)': 0.9256454618996856,
 'R2_gap': 0.02816372199098327}